# Chinese -> Tao: fine-tuned NLLB draft + Gemma 4 RAG post-editing (local)

Part 2 of 2. Run this notebook on the **local machine** (RTX 4060). Part 1 (`tao_nllb_finetune.ipynb`, Colab) produces the fine-tuned NLLB model in `models/nllb-tao-ft/`.

Same workflow as `eval/tetun_bislama/`, with the draft system replaced by the fine-tuned NLLB model. This notebook loads the model, translates a sentence, runs NLLB -> Gemma 4 post-editing on data **B** with retrieval from **C** (translation memory) and **D** (glossary), and scores with chrF++ / BLEU.

**Local environment**: install a CUDA build of PyTorch first (check `torch.cuda.is_available()`), then run the pip cell below.

**Expected data layout** (relative to the repo root):

```
data/finetune/   A  fine-tuning pairs (only used here for the leak check)
data/eval/       B  evaluation pairs
data/memory/     C  translation memory (retrieval pool)
data/glossary/   D  Chinese-Tao glossary
models/nllb-tao-ft/   fine-tuned model from Part 1 (or models/nllb-tao-ft.zip)
```

Each data folder may hold `.json`, `.jsonl` or `.csv` files.
- A, B, C: records with `chinese_sentence` and `tao_sentence` (aliases `zh` / `tao` also work). Any other keys, such as `source`, are ignored.
- D: records with `zh` and `tao` (aliases `key` / `value` work too).

Keep B out of A and C. The leak check in section 1 warns about exact duplicates; for Bible-style data, also split by book/chapter. Set `DEEPINFRA_API_KEY` in the environment or in a `.env` file.

## 0. Setup

In [ ]:
%pip install -q transformers sentencepiece sacrebleu jieba rank-bm25 openai python-dotenv pandas tqdm

In [ ]:
import os, json, csv, re, time, hashlib, shutil, threading
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

ROOT = Path("../..").resolve()   # repo root, when this notebook is run from eval/tao/

FINETUNE_DIR = ROOT / "data" / "finetune"   # A
EVAL_DIR     = ROOT / "data" / "eval"       # B
TM_DIR       = ROOT / "data" / "memory"     # C
GLOSSARY_DIR = ROOT / "data" / "glossary"   # D
MODEL_DIR    = ROOT / "models" / "nllb-tao-ft"
CACHE_DIR    = ROOT / "cache"
RESULTS_DIR  = ROOT / "results"
for d in (CACHE_DIR, RESULTS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# --- languages ---
SRC_LANG = "zho_Hant"        # use "zho_Hans" for simplified Chinese
TGT_LANG = "tao_Latn"        # NOT in NLLB-200: added as a new token in tao_nllb_finetune.ipynb
TGT_NAME = "Tao"

# --- NLLB inference ---
MAX_LEN = 128

# --- inference / evaluation ---
NUM_BEAMS = 4
MAX_NEW_TOKENS = 200
MODE = "post_edit"           # "post_edit" (NLLB draft + Gemma edit) or "translate_only" (Gemma few-shot, no draft)
TOP_N_SIMILAR = 10
EXCLUDE_EXACT_MATCH = True   # drop TM entries whose Chinese text equals the test sentence (honest evaluation)
EVAL_LIMIT = None            # e.g. 50 for a quick smoke test

# --- Gemma 4 via DeepInfra (OpenAI-compatible) ---
GEMMA_MODEL = "google/gemma-4-26B-A4B-it"
DEEPINFRA_BASE_URL = "https://api.deepinfra.com/v1/openai"
TEMPERATURE = 0.5
MAX_WORKERS = 8

## 1. Load and check the data

In [ ]:
def read_records(path: Path):
    ext = path.suffix.lower()
    if ext == ".jsonl":
        with open(path, encoding="utf-8") as f:
            return [json.loads(line) for line in f if line.strip()]
    if ext == ".json":
        with open(path, encoding="utf-8") as f:
            data = json.load(f)
        return data if isinstance(data, list) else list(data.values())
    if ext == ".csv":
        with open(path, encoding="utf-8", newline="") as f:
            return list(csv.DictReader(f))
    return []

def _first(row, keys):
    for k in keys:
        if row.get(k):
            return str(row[k]).strip()
    return ""

def read_folder(folder: Path):
    if not folder.exists():
        raise FileNotFoundError(f"{folder} not found")
    rows = []
    for p in sorted(folder.iterdir()):
        rows += read_records(p)
    if not rows:
        raise ValueError(f"no .json/.jsonl/.csv records in {folder}")
    return rows

def load_pairs(folder: Path):
    """-> list of {'zh', 'tao'}; empty and duplicate pairs dropped."""
    out, seen = [], set()
    for r in read_folder(folder):
        zh = _first(r, ("chinese_sentence", "zh", "chinese"))
        tao = _first(r, ("tao_sentence", "tao", "yami"))
        if zh and tao and (zh, tao) not in seen:
            seen.add((zh, tao))
            out.append({"zh": zh, "tao": tao})
    return out

def load_glossary(folder: Path):
    out = []
    for r in read_folder(folder):
        zh = _first(r, ("zh", "key", "chinese"))
        tao = _first(r, ("tao", "value", "yami"))
        if zh and tao:
            out.append({"zh": zh, "tao": tao})
    return out

In [ ]:
data_a = load_pairs(FINETUNE_DIR)
data_b = load_pairs(EVAL_DIR)
data_c = load_pairs(TM_DIR)
data_d = load_glossary(GLOSSARY_DIR)
print(f"A fine-tune: {len(data_a)} | B eval: {len(data_b)} | C memory: {len(data_c)} | D glossary: {len(data_d)}")

# leakage check: evaluation sentences that also appear in A or C
b_zh = {x["zh"] for x in data_b}
print("B sentences also in A:", len(b_zh & {x["zh"] for x in data_a}))
print("B sentences also in C:", len(b_zh & {x["zh"] for x in data_c}),
      "(excluded from retrieval)" if EXCLUDE_EXACT_MATCH else "(NOT excluded: scores will be inflated)")
data_b[0]

## 2. Locate the fine-tuned model

Produced by `tao_nllb_finetune.ipynb` on Colab. If you downloaded the zip instead of syncing Drive, put it at `models/nllb-tao-ft.zip`.

In [ ]:
def model_is_cached() -> bool:
    return (MODEL_DIR / "config.json").exists() and (MODEL_DIR / "tokenizer_config.json").exists()

zip_path = ROOT / "models" / "nllb-tao-ft.zip"
if not model_is_cached() and zip_path.exists():
    shutil.unpack_archive(zip_path, MODEL_DIR)
assert model_is_cached(), f"No fine-tuned model at {MODEL_DIR}; run tao_nllb_finetune.ipynb on Colab first."
print("Model cache OK:", MODEL_DIR)

## 3. Load the fine-tuned NLLB and translate

All translations are cached on disk (`cache/`), so re-runs are free.

In [ ]:
class JsonCache:
    def __init__(self, path: Path):
        self.path, self.lock, self.dirty = path, threading.Lock(), 0
        self.data = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}

    def get(self, key):
        return self.data.get(key)

    def set(self, key, value, save_every=20):
        with self.lock:
            self.data[key] = value
            self.dirty += 1
            if self.dirty >= save_every:
                self.save()

    def save(self):
        self.path.write_text(json.dumps(self.data, ensure_ascii=False, indent=1), encoding="utf-8")
        self.dirty = 0

def md5(*parts) -> str:
    return hashlib.md5(json.dumps(parts, ensure_ascii=False).encode()).hexdigest()

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
mt_tok = AutoTokenizer.from_pretrained(MODEL_DIR, src_lang=SRC_LANG)
mt_tok.tgt_lang = TGT_LANG
mt_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR).to(device).eval()
tao_id = mt_tok.convert_tokens_to_ids(TGT_LANG)
mt_cache = JsonCache(CACHE_DIR / "nllb_tao_ft.json")
print("device:", device, "| tao_Latn id:", tao_id)

@torch.inference_mode()
def _generate(texts):
    enc = mt_tok(texts, return_tensors="pt", padding=True, truncation=True, max_length=MAX_LEN).to(device)
    out = mt_model.generate(**enc, forced_bos_token_id=tao_id, num_beams=NUM_BEAMS, max_new_tokens=MAX_NEW_TOKENS)
    return mt_tok.batch_decode(out, skip_special_tokens=True)

def translate_many(texts, batch_size=16):
    """Chinese -> Tao with the fine-tuned NLLB; returns a list aligned with `texts`."""
    key = lambda t: md5(str(MODEL_DIR), NUM_BEAMS, t)
    todo = sorted({t for t in texts if mt_cache.get(key(t)) is None}, key=len)
    for i in tqdm(range(0, len(todo), batch_size), desc="NLLB", disable=not todo):
        batch = todo[i:i + batch_size]
        for t, hyp in zip(batch, _generate(batch)):
            mt_cache.set(key(t), hyp)
    mt_cache.save()
    return [mt_cache.get(key(t)) for t in texts]

def translate_one(text: str) -> str:
    return translate_many([text])[0]

In [ ]:
demo = "後來埃及和迦南全地遭遇饑荒。"
print("zh :", demo)
print("tao:", translate_one(demo))

## 4. Retrieval: translation memory C and glossary D

Same strategy as `tetun_bislama/`: BM25 over the source side of past translations, top N neighbours, plus glossary entries found in the sentence. Differences: Chinese is segmented with `jieba` instead of spaCy lemmas, and glossary matching is a substring match because Chinese has no word spaces.

In [ ]:
import jieba
from rank_bm25 import BM25Okapi

_PUNCT = re.compile(r"[\W_]+", re.UNICODE)
_STOP = set("的了和是在也就都而及與与以之有不他她它我你們们")

def zh_tokens(s: str):
    toks = [t for t in jieba.lcut(s) if t.strip() and not _PUNCT.fullmatch(t) and t not in _STOP]
    return toks or [c for c in s if c.strip()]

tm_tokens = [zh_tokens(x["zh"]) for x in data_c]
bm25 = BM25Okapi(tm_tokens)
glossary_sorted = sorted(data_d, key=lambda e: -len(e["zh"]))

def similar_entries(zh: str, top_n: int = TOP_N_SIMILAR):
    scores = bm25.get_scores(zh_tokens(zh))
    out = []
    for i in np.argsort(scores)[::-1]:
        if scores[i] <= 0 or len(out) >= top_n:
            break
        if EXCLUDE_EXACT_MATCH and data_c[i]["zh"] == zh:
            continue
        out.append(data_c[i])
    return out

def glossary_hits(zh: str, limit: int = 30):
    hits, seen = [], set()
    for e in glossary_sorted:
        if e["zh"] in zh and e["zh"] not in seen:
            seen.add(e["zh"])
            hits.append(e)
    return hits[:limit]

q = data_b[0]["zh"]
print(q)
for e in similar_entries(q)[:3]:
    print("  ", e["zh"], "->", e["tao"])
print("glossary:", glossary_hits(q))

## 5. Gemma 4 client (DeepInfra)

Responses are cached by `md5(model, temperature, messages)`, so changing the model or temperature never returns stale answers. The system prompt is folded into the user turn, which works for any chat template.

In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

api_key = os.environ.get("DEEPINFRA_API_KEY")
assert api_key, "Set DEEPINFRA_API_KEY (environment, .env, or Colab secret)"

llm = OpenAI(api_key=api_key, base_url=DEEPINFRA_BASE_URL)
llm_cache = JsonCache(CACHE_DIR / "gemma_deepinfra.json")

def chat(system: str, user: str, retries: int = 3) -> str:
    k = md5(GEMMA_MODEL, TEMPERATURE, system, user)
    hit = llm_cache.get(k)
    if hit is not None:
        return hit
    for attempt in range(retries):
        try:
            r = llm.chat.completions.create(
                model=GEMMA_MODEL, temperature=TEMPERATURE,
                messages=[{"role": "user", "content": f"{system}\n\n{user}"}],
            )
            out = r.choices[0].message.content.strip()
            llm_cache.set(k, out)
            return out
        except Exception as e:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)

## 6. Prompts

Post-edit prompt mirrors `construct_prompt_post_edit` in `translator_utils.py`: glossary entries, then past translations as `Chinese / MT / reference` triples, then the new sentence with its draft. `translate_only` mirrors the few-shot translate mode (no draft).

In [ ]:
SYSTEM_POST_EDIT = (
    f"You are an expert translator. I am going to give you relevant glossary entries and relevant past translations. "
    f"Each past translation has three parts: the Chinese source, a machine translation of it into {TGT_NAME}, and the "
    f"human {TGT_NAME} reference translation. Then I will give you a new Chinese sentence and its machine translation. "
    f"Correct the machine translation so that it is an accurate, fluent {TGT_NAME} translation, following the "
    f"vocabulary, spelling and style of the reference translations and the glossary. Only change what is wrong. "
    f"Put the corrected {TGT_NAME} translation inside <{TGT_NAME.lower()}>...</{TGT_NAME.lower()}> tags and write nothing else."
)
SYSTEM_TRANSLATE = (
    f"You are an expert translator. I am going to give you some example pairs of Chinese text and their {TGT_NAME} "
    f"translation, then a new Chinese sentence. Translate the new sentence into {TGT_NAME}. "
    f"Put the translation inside <{TGT_NAME.lower()}>...</{TGT_NAME.lower()}> tags and write nothing else."
)

def build_post_edit_prompt(zh, neighbours, neighbour_mt, gloss, draft):
    u = ""
    if gloss:
        u += "<glossary entries>\n" + "".join(f"no {i}: {g['zh']} -> {g['tao']}\n" for i, g in enumerate(gloss)) + "</glossary entries>\n\n"
    u += "<past translations>\n"
    for n, mt in zip(neighbours, neighbour_mt):
        u += f"Chinese: {n['zh']}\nMT: {mt}\n{TGT_NAME}: {n['tao']}\n\n"
    u += "</past translations>\n\nText to translate:\n"
    u += f"Chinese: {zh}\nMT: {draft}\n{TGT_NAME}: "
    return SYSTEM_POST_EDIT, u

def build_translate_prompt(zh, neighbours):
    u = "".join(f"Chinese: {n['zh']}\n{TGT_NAME}: {n['tao']}\n\n" for n in neighbours)
    u += f"Chinese: {zh}\n{TGT_NAME}: "
    return SYSTEM_TRANSLATE, u

_TAG = TGT_NAME.lower()
def strip_tags(resp: str) -> str:
    m = re.search(rf"<{_TAG}>(.*?)</{_TAG}>", resp, re.S | re.I)
    return (m.group(1) if m else re.sub(rf"</?{_TAG}>", "", resp, flags=re.I)).strip()

## 7. Try one sentence end to end

In [ ]:
def post_edit_one(zh: str, verbose: bool = False) -> dict:
    neighbours = similar_entries(zh)
    gloss = glossary_hits(zh)
    texts = [zh] + [n["zh"] for n in neighbours]
    draft, *neighbour_mt = translate_many(texts)
    system, user = build_post_edit_prompt(zh, neighbours, neighbour_mt, gloss, draft)
    if verbose:
        print(system, "\n\n", user, "\n")
    return {"mt": draft, "ape": strip_tags(chat(system, user))}

r = post_edit_one(data_b[0]["zh"], verbose=True)
print("NLLB draft :", r["mt"])
print("Gemma edit :", r["ape"])
print("Reference  :", data_b[0]["tao"])

## 8. Run on evaluation set B

In [ ]:
test = data_b[:EVAL_LIMIT] if EVAL_LIMIT else data_b

# retrieval for every test sentence, then one batched NLLB pass over test + neighbour sentences
retrieved = [(similar_entries(x["zh"]), glossary_hits(x["zh"])) for x in tqdm(test, desc="retrieve")]
all_zh = list(dict.fromkeys([x["zh"] for x in test] + [n["zh"] for nb, _ in retrieved for n in nb]))
translate_many(all_zh)
mt_drafts = translate_many([x["zh"] for x in test])

def run_one(i):
    zh = test[i]["zh"]
    neighbours, gloss = retrieved[i]
    if MODE == "post_edit":
        system, user = build_post_edit_prompt(zh, neighbours, translate_many([n["zh"] for n in neighbours]), gloss, mt_drafts[i])
    else:
        system, user = build_translate_prompt(zh, neighbours)
    return i, strip_tags(chat(system, user))

hyps, failures = {}, 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    futures = [ex.submit(run_one, i) for i in range(len(test))]
    for f in tqdm(as_completed(futures), total=len(futures), desc=f"Gemma ({MODE})"):
        try:
            i, out = f.result()
            hyps[i] = out
        except Exception as e:
            failures += 1
            print("failed:", e)
llm_cache.save()

df = pd.DataFrame({
    "zh": [x["zh"] for x in test],
    "ref": [x["tao"] for x in test],
    "mt": mt_drafts,
    "llm": [hyps.get(i, "") for i in range(len(test))],
})
print(f"{len(df)} lines, {failures} failed calls")
df.head()

## 9. Evaluation

chrF++ (`word_order=2`, as in the other notebooks) and BLEU on all lines where the LLM call succeeded. `mt` is the fine-tuned NLLB draft on its own; `llm` is the Gemma output.

In [ ]:
from sacrebleu import corpus_chrf, corpus_bleu, sentence_chrf

ok = df[df["llm"] != ""].copy()
refs = [ok["ref"].tolist()]

def score(col):
    return {
        "chrF++": corpus_chrf(ok[col].tolist(), refs, word_order=2).score,
        "BLEU": corpus_bleu(ok[col].tolist(), refs).score,
    }

summary = pd.DataFrame({
    "NLLB fine-tuned (draft)": score("mt"),
    f"{'NLLB + Gemma 4 post-edit' if MODE == 'post_edit' else 'Gemma 4 few-shot'}": score("llm"),
}).T.round(2)
summary["diff chrF++"] = (summary["chrF++"] - summary.iloc[0]["chrF++"]).round(2)
display(summary)

ok["chrf_mt"] = [sentence_chrf(h, [r], word_order=2).score for h, r in zip(ok["mt"], ok["ref"])]
ok["chrf_llm"] = [sentence_chrf(h, [r], word_order=2).score for h, r in zip(ok["llm"], ok["ref"])]
better, worse = (ok["chrf_llm"] > ok["chrf_mt"]).sum(), (ok["chrf_llm"] < ok["chrf_mt"]).sum()
print(f"n={len(ok)}  LLM better on {better}, worse on {worse}, tied on {len(ok) - better - worse} sentences")

In [ ]:
for _, r in ok.sample(min(5, len(ok)), random_state=0).iterrows():
    print("zh  :", r["zh"])
    print("MT  :", r["mt"])
    print("LLM :", r["llm"])
    print("REF :", r["ref"])
    print()

In [ ]:
stamp = time.strftime("%Y%m%d-%H%M%S")
ok.to_csv(RESULTS_DIR / f"tao_eval_{MODE}_{stamp}.csv", index=False)
with open(RESULTS_DIR / "tao_eval_summary.jsonl", "a", encoding="utf-8") as f:
    f.write(json.dumps({
        "time": stamp, "mode": MODE, "n": len(ok), "llm": GEMMA_MODEL, "nllb": str(MODEL_DIR.name),
        "top_n": TOP_N_SIMILAR, "chrf_mt": round(summary.iloc[0]["chrF++"], 2),
        "chrf_llm": round(summary.iloc[1]["chrF++"], 2),
        "bleu_mt": round(summary.iloc[0]["BLEU"], 2), "bleu_llm": round(summary.iloc[1]["BLEU"], 2),
    }) + "\n")
print("saved to", RESULTS_DIR)